# Colab GPU Chat UI (HF形式そのまま)

- GGUF不要で、Colab GPU上でチャット推論を行うための最小構成です。
- `MODEL_SOURCE` を切り替えることで、以下の3パターンに対応します。
  - `base`: ベースモデル
  - `merged`: すでにマージ済みモデル
  - `adapter_merge`: ベース + LoRAアダプタをその場でマージ

In [ ]:
# 必要なときだけ True にして実行してください（実行後はランタイム再起動）
# vLLMチャット用途の最小セット（StructEval依存なし）
RUN_INSTALL = True

if RUN_INSTALL:
    import subprocess

    cmds = [
        # 競合しやすい主要パッケージを先に外してから最小セットを入れる
        'pip uninstall -y protobuf huggingface_hub transformers tokenizers vllm',
        'pip install --no-cache-dir "protobuf==5.29.3"',
        'pip install --no-cache-dir '
        '"torch==2.9.0" '
        '"triton==3.5.0" '
        '"huggingface-hub>=0.36.0" '
        '"tokenizers>=0.22.0" '
        '"vllm>=0.13.0" '
        '"gradio>=4.0.0" '
        '"accelerate" '
        '"peft"',
        # Qwen3.5(qwen3_5)はtransformersメインラインが必要な場合がある
        'pip install --no-cache-dir --upgrade "git+https://github.com/huggingface/transformers.git"',
        'python3 -c "import google.protobuf, vllm, transformers, huggingface_hub, gradio; '
        "from transformers.models.auto.configuration_auto import CONFIG_MAPPING; "
        "print('protobuf', google.protobuf.__version__); "
        "print('vllm', vllm.__version__); "
        "print('transformers', transformers.__version__); "
        "print('huggingface_hub', huggingface_hub.__version__); "
        "print('qwen3_5_supported', 'qwen3_5' in CONFIG_MAPPING)" '"',
    ]
    for c in cmds:
        subprocess.check_call(c, shell=True)
    print("✅ setup finished. Please restart runtime now.")



In [ ]:
import os

# -----------------------------
# Config
# -----------------------------
MODEL_SOURCE = "base"  # "base" | "merged" | "adapter_merge"

# HF形式モデルID
BASE_MODEL_ID = "Qwen/Qwen3.5-4B"
MERGED_MODEL_ID = ""
ADAPTER_ID = ""

# 推論設定（Transformers fallback向けに控えめ推奨）
MAX_NEW_TOKENS = 512
MAX_INPUT_TOKENS = 2048
MAX_HISTORY_TURNS = 3
TEMPERATURE = 0.0
TOP_P = 1.0

# merge一時保存先（adapter_merge時）
MERGED_LOCAL_DIR = "/content/merged_for_chat"



In [ ]:
import gc
import json
from pathlib import Path

from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.models.auto.configuration_auto import CONFIG_MAPPING
import google.protobuf
import torch


def detect_model_type(path_or_repo: str):
    # Local merged dir を優先して config.json を読む
    local_cfg = Path(path_or_repo) / "config.json"
    if local_cfg.exists():
        return json.loads(local_cfg.read_text()).get("model_type")

    # HF repo の場合はキャッシュ済み config.json を読む
    from huggingface_hub import hf_hub_download

    cfg_path = hf_hub_download(path_or_repo, filename="config.json")
    return json.loads(Path(cfg_path).read_text()).get("model_type")


def validate_transformers_arch_support(path_or_repo: str):
    model_type = detect_model_type(path_or_repo)
    if model_type is None:
        print("[WARN] model_type could not be detected from config.json")
        return None

    if model_type not in CONFIG_MAPPING:
        import transformers

        raise RuntimeError(
            f"Installed transformers={transformers.__version__} does not support model_type={model_type}. "
            "Run install cell (RUN_INSTALL=True) and restart runtime."
        )

    print(f"[INFO] model_type={model_type} is supported by transformers")
    return model_type


def resolve_model_path_and_tokenizer():
    if MODEL_SOURCE == "base":
        model_id = BASE_MODEL_ID
        print(f"[INFO] Using base model: {model_id}")
        tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
        return model_id, tokenizer

    if MODEL_SOURCE == "merged":
        model_id = MERGED_MODEL_ID
        print(f"[INFO] Using merged model: {model_id}")
        tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
        return model_id, tokenizer

    if MODEL_SOURCE == "adapter_merge":
        from peft import PeftModel

        print(f"[INFO] Loading base model for merge: {BASE_MODEL_ID}")
        # vLLM初期化時のGPU競合を避けるため、マージはCPU上で実行
        base_model = AutoModelForCausalLM.from_pretrained(
            BASE_MODEL_ID,
            torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
            device_map="cpu",
            trust_remote_code=True,
        )
        tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)

        print(f"[INFO] Loading adapter: {ADAPTER_ID}")
        lora_model = PeftModel.from_pretrained(base_model, ADAPTER_ID)

        print("[INFO] Merging adapter...")
        merged_model = lora_model.merge_and_unload()

        os.makedirs(MERGED_LOCAL_DIR, exist_ok=True)
        merged_model.save_pretrained(MERGED_LOCAL_DIR)
        tokenizer.save_pretrained(MERGED_LOCAL_DIR)

        del base_model, lora_model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        print(f"[INFO] Merged model saved to: {MERGED_LOCAL_DIR}")
        return MERGED_LOCAL_DIR, tokenizer

    raise ValueError("MODEL_SOURCE must be one of: base, merged, adapter_merge")


model_path, tokenizer = resolve_model_path_and_tokenizer()
print(f"[INFO] Model path resolved: {model_path}")
model_type = validate_transformers_arch_support(model_path)

# 本番環境合わせ: protobuf 5.29.3 を期待
pb_ver = google.protobuf.__version__
if pb_ver != "5.29.3":
    raise RuntimeError(
        f"Incompatible protobuf version: {pb_ver}. "
        "Please run install cell and restart runtime."
    )

if torch.cuda.is_available():
    # L4でのTransformers推論速度改善
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.set_float32_matmul_precision("high")

# vLLMはimport前に環境変数を設定
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"
os.environ["VLLM_LOGGING_LEVEL"] = "INFO"
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "1"

# tokenizerはベース側を明示（mergedフォルダのtokenizer由来不整合を回避）
tokenizer_path_for_vllm = BASE_MODEL_ID if MODEL_SOURCE == "adapter_merge" else model_path

# vLLM起動フォールバック（OOM/初期化失敗を段階的に回避）
_try_cfgs = [
    {"max_model_len": 4096, "gpu_memory_utilization": 0.85},
    {"max_model_len": 3072, "gpu_memory_utilization": 0.80},
    {"max_model_len": 2048, "gpu_memory_utilization": 0.72},
]
llm = None
vllm_available = False
fallback_transformers_model = None
last_err = None
SamplingParams = None

# qwen3_5 + latest transformers では vllm との組み合わせが壊れることがあるため安全にフォールバック
vllm_import_ok = False
try:
    from vllm import LLM, SamplingParams
    vllm_import_ok = True
except Exception as e:
    last_err = e
    print(f"[WARN] vLLM import failed: {type(e).__name__}: {e}")

if vllm_import_ok:
    for i, cfg in enumerate(_try_cfgs, 1):
        try:
            print(
                f"[INFO] vLLM init try {i}/{len(_try_cfgs)} "
                f"(max_model_len={cfg['max_model_len']}, gpu_mem={cfg['gpu_memory_utilization']})"
            )
            llm = LLM(
                model=model_path,
                tokenizer=tokenizer_path_for_vllm,
                trust_remote_code=True,
                tensor_parallel_size=1,
                enforce_eager=True,
                max_model_len=cfg["max_model_len"],
                gpu_memory_utilization=cfg["gpu_memory_utilization"],
                disable_log_stats=True,
            )
            print("[INFO] vLLM loaded.")
            vllm_available = True
            break
        except Exception as e:
            last_err = e
            print(f"[WARN] vLLM init failed on try {i}: {type(e).__name__}: {e}")

if not vllm_available:
    print(f"[WARN] Using Transformers backend. vLLM last_error={last_err}")
    fallback_transformers_model = AutoModelForCausalLM.from_pretrained(
        model_path,
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        device_map="auto",
        trust_remote_code=True,
    )
    fallback_transformers_model.eval()
    print(
        "[INFO] Transformers fallback model loaded on device:",
        getattr(fallback_transformers_model, "device", "unknown"),
    )

if torch.cuda.is_available():
    print("[INFO] CUDA is available:", torch.cuda.get_device_name(0))
else:
    print("[WARN] CUDA is NOT available. Running on CPU.")



In [ ]:
import re
import gradio as gr


def _strip_thinking(text: str) -> str:
    # Qwen系が出す思考トークンをUI表示前に除去
    text = re.sub(r"<think>[\s\S]*?</think>", "", text)
    return text.strip()


def chat_fn(message, history):
    # 履歴を短く保って毎ターンの前処理・推論を軽くする
    recent_history = history[-MAX_HISTORY_TURNS:] if MAX_HISTORY_TURNS > 0 else history

    messages = [
        {
            "role": "system",
            "content": "You are a concise assistant. Do not output chain-of-thought or hidden reasoning. Return only the final answer.",
        }
    ]
    for user_text, assistant_text in recent_history:
        messages.append({"role": "user", "content": user_text})
        messages.append({"role": "assistant", "content": assistant_text})
    messages.append({"role": "user", "content": message})

    # transformersの版差で enable_thinking 引数の有無があるため吸収
    try:
        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
    except TypeError:
        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

    if vllm_available:
        sampling = SamplingParams(
            max_tokens=MAX_NEW_TOKENS,
            temperature=TEMPERATURE,
            top_p=TOP_P,
        )
        outs = llm.generate([prompt], sampling)
        text = outs[0].outputs[0].text.strip() if outs and outs[0].outputs else ""
    else:
        import torch

        tok = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_INPUT_TOKENS)
        inputs = {k: v.to(fallback_transformers_model.device) for k, v in tok.items()}

        with torch.no_grad():
            outputs = fallback_transformers_model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=(TEMPERATURE > 0.0),
                temperature=TEMPERATURE,
                top_p=TOP_P,
                use_cache=True,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

        gen_ids = outputs[0][inputs["input_ids"].shape[1]:]
        text = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

    return _strip_thinking(text)


demo = gr.ChatInterface(
    fn=chat_fn,
    title="Colab GPU Chat (HF model)",
    description="GGUF不要。Colab GPU上で直接チャット推論。",
)

# share=True で外部URL発行
demo.launch(share=True, debug=False)

